# Day 4-5 | Statistical tests, time features, and final insights

All statistics are descriptive or associational. The employee-level file is pseudonymized; do not use these findings for individual decisions. AS_OF_DATE matches Day 2 (2026-10-29).

## 1. Load clean data and shared analysis date

In [ ]:
from pathlib import Path
import pandas as pd,numpy as np
import matplotlib.pyplot as plt,seaborn as sns
from scipy.stats import ttest_ind,f_oneway,chi2_contingency,levene,kruskal,spearmanr,mannwhitneyu
sns.set_theme(style="whitegrid")
df=pd.read_csv("../data/interim/cleaned_day2.csv")
for c in ["hire_date","termination_date"]:df[c]=pd.to_datetime(df[c],errors="coerce")
AS_OF_DATE=pd.Timestamp("2026-10-29")
assert len(df)==5500
print("Rows",len(df),"AS_OF_DATE",AS_OF_DATE)

## 2. Welch t-test: satisfaction by sex
H0: population mean satisfaction is equal for Female and Male groups. H1: means differ. Welch’s test does not assume equal variances. Include effect size with the p-value.

In [ ]:
female=df.loc[df.sex.eq("Female"),"employee_satisfaction"].dropna();male=df.loc[df.sex.eq("Male"),"employee_satisfaction"].dropna()
tres=ttest_ind(female,male,equal_var=False)
pooled=np.sqrt(((len(female)-1)*female.var(ddof=1)+(len(male)-1)*male.var(ddof=1))/(len(female)+len(male)-2))
d=(female.mean()-male.mean())/pooled
print({"n_female":len(female),"mean_female":female.mean(),"n_male":len(male),"mean_male":male.mean(),"t":tres.statistic,"p":tres.pvalue,"cohens_d":d})

## 3. Salary differences by department
H0: salary distributions are the same across departments. Check variance homogeneity; report classical ANOVA, Levene, and Kruskal-Wallis. The nonparametric result tests distributions, not strictly medians. Pairwise Mann-Whitney comparisons use Holm adjustment.

In [ ]:
groups={k:v.salary.dropna() for k,v in df.groupby("department") if v.salary.notna().sum()>1}
print("Group medians/counts",{k:(len(v),v.median()) for k,v in groups.items()})
print("Levene",levene(*groups.values()))
print("One-way ANOVA",f_oneway(*groups.values()))
kw=kruskal(*groups.values()); n=sum(map(len,groups.values())); k=len(groups)
epsilon=max(0,(kw.statistic-k+1)/(n-k))
print("Kruskal-Wallis",kw,"epsilon_squared",epsilon)
pairs=[];names=list(groups)
for i in range(len(names)):
 for j in range(i+1,len(names)):
  res=mannwhitneyu(groups[names[i]],groups[names[j]],alternative="two-sided")
  pairs.append((names[i],names[j],res.statistic,res.pvalue))
rawp=np.array([x[3] for x in pairs]); order=np.argsort(rawp); adjusted=np.empty(len(rawp)); running=0.0
for rank,idx in enumerate(order):
    running=max(running,(len(rawp)-rank)*rawp[idx]); adjusted[idx]=min(running,1.0)
adj=adjusted
display(pd.DataFrame([(*pairs[i],adj[i]) for i in range(len(pairs))],columns=["group_1","group_2","U","p_raw","p_holm"]).sort_values("p_holm").head(20))

## 4. Chi-square: work mode and employment status
H0: work mode and employment status are independent. Exclude Missing work mode from this substantive test, but retain it in the dataset and report its count. Check expected frequencies and report Cramer’s V.

In [ ]:
known=df[df.work_mode.ne("Missing")]
tab=pd.crosstab(known.work_mode,known.employment_status)
chi2,p,dof,expected=chi2_contingency(tab)
cramers=np.sqrt(chi2/(tab.to_numpy().sum()*min(tab.shape[0]-1,tab.shape[1]-1)))
print(tab);print({"chi2":chi2,"dof":dof,"p":p,"min_expected":expected.min(),"cramers_v":cramers,"excluded_missing_work_mode":int(df.work_mode.eq("Missing").sum())})

## 5. Additional association: engagement and satisfaction

In [ ]:
rho=spearmanr(df.engagement_score,df.employee_satisfaction,nan_policy="omit")
print("Spearman rho, p, n",rho.statistic,rho.pvalue,int(df[["engagement_score","employee_satisfaction"]].dropna().shape[0]))

## 6. Time features and trends
Use a single AS_OF_DATE for every active-employee tenure estimate. The 2026 hiring year is partial; future hires were nulled in Day 2.

In [ ]:
hire=df.hire_date;term=df.termination_date
df["hire_year"]=hire.dt.year;df["hire_month"]=hire.dt.month;df["hire_quarter"]=hire.dt.quarter;df["hire_weekday"]=hire.dt.day_name()
df["tenure_days"]=(term.fillna(AS_OF_DATE)-hire).dt.days;df["tenure_years"]=(df.tenure_days/365.2425).round(2)
df["is_terminated"]=df.terminated.astype("int8");df["high_absence_flag"]=df.absences.ge(df.absences.quantile(.75)).astype("int8");df["low_satisfaction_flag"]=df.employee_satisfaction.le(2).astype("int8");df["is_ever_late"]=df.days_late_last_30.gt(0).astype("int8")
annual=df.groupby("hire_year").size();display(annual.rename("hires"))
annual.plot(marker="o",figsize=(9,4),title="Hires by year");plt.tight_layout();plt.savefig("../reports/figures/hires_by_year.png",dpi=160);plt.show()
monthly=df.groupby("hire_month").size().reindex(range(1,13));monthly.plot(kind="bar",figsize=(9,4),title="Hires by month");plt.tight_layout();plt.savefig("../reports/figures/hires_by_month.png",dpi=160);plt.show()
df.tenure_years.hist(bins=30,figsize=(8,4));plt.title("Tenure distribution");plt.tight_layout();plt.savefig("../reports/figures/tenure_distribution.png",dpi=160);plt.show()

## 7. Final insights generated from observed data
This cell prints candidate findings with actual sample values. Review wording and caveats before presentation; do not describe the terminated share as an annual turnover rate.

In [ ]:
status_pct=(df.employment_status.value_counts(normalize=True)*100).round(2)
print("Employment status %",status_pct.to_dict())
print("Engagement-satisfaction Pearson",df[["engagement_score","employee_satisfaction"]].corr().iloc[0,1])
print("Top recruitment sources",df.recruitment_source.value_counts().head().to_dict())
print("Median salary by department",df.groupby("department").salary.median().sort_values(ascending=False).to_dict())
print("Median engagement by status",df.groupby("employment_status").engagement_score.median().to_dict())
print("Median satisfaction by status",df.groupby("employment_status").employee_satisfaction.median().to_dict())
print("Top termination reasons",df.loc[df.terminated.eq(1)&df.termination_reason.ne("N/A - Still Employed"),"termination_reason"].value_counts().head(10).to_dict())
print("Hires by year",df.groupby("hire_year").size().to_dict())
print("Absence over-60 flagged",int(df.absences_over_60_flag.sum()),"Salary corrections",int(df.salary_corrected_flag.sum()))
print("Terminated share is not an annual turnover rate; dataset has no defined annual denominator.")
df.to_csv("../data/processed/final_cleaned_day4.csv",index=False)

## Limitations and privacy
Names and exact DOB are excluded from the bundled raw/interim/processed files. Employee IDs are random pseudonyms, which still require care. DOB-based age analysis is omitted because two-digit dates are ambiguous and exact DOB is sensitive. Duplicate audit has no direct identifiers. Results are observational, the 2026 period is incomplete, and AS_OF_DATE is an assumed analysis reference date.